# Fine-tuning RXNGraphormer on Yield Dataset – User Guide

## 1. Overview

This guide describes how to fine-tune a pre-trained **RXNGraphormer** model on an external yield dataset using the provided grid search script. The workflow ensures **reproducibility** by saving all configurations, logs, and results for each experiment.

---

## 2. Requirements

- External dataset in CSV format with:
  - `reaction_smiles`
  - `yield`
- Pre-trained RXNGraphormer checkpoint (configured in `train_model.py`)

---

## 3. Workflow

The script performs:

- Loading a base configuration (`CONFIG`)
- Generating hyperparameter combinations (`SPACE`)
- Automatically selecting available GPU
- Running training via:
  ```bash
  python train_model.py --config_json <config>
  ```
## 4. Usage
**Step 1: Set configuration***
```python
CONFIG = "./config/Test/sulfoxonium_seed68909_parameters.json"
```

**Step 2: Define search space**
```python
SPACE = {
    "train_ratio": [0.8],
    "valid_ratio": [0.2],
    "seed": [42],
    "learning_rate": [0.4, 0.5],
    "batch_size": [32, 64],
    "warmup_step": [3000],
}
```

**Step 3: Run**
```bash
 cd reproduction
```
```python
python finetune_grid_search.py
```
## 5. Output

- `results.csv`: summary of all experiments (MAE, R²)

- `grid_configs/`: saved configurations for each run

- `experiment_logs/`: training logs

- `model_path/`: trained model checkpoints

Each experiment can be reproduced by:

```bash
python train_model.py --config_json <config.json>
```
## 6. Example

### 6.1 Sulfoxonium Dataset

This example performs a grid search over key hyperparameters (learning rate, batch size, and warmup steps) on the Sulfoxonium dataset.

All experiments are automatically:

- executed on available GPUs,

- logged with full configurations,

- evaluated using Test MAE and R².

The returned best_results table is sorted by Test MAE, allowing direct identification of the best-performing configuration. Each experiment can be fully reproduced using the saved configuration files.


In [1]:
from reproduction.finetune_grid_search import run_grid_search
CONFIG = "./config/Test/sulfoxonium_seed68909_parameters.json"
SPACE = {
    "train_ratio": [0.7], # 0.8
    "valid_ratio": [0.3], # 0.2
    "seed": [42],
    "tag": [f"seed{42}"],
    "learning_rate": [0.4, 0.5],
    "batch_size": [32, 64],
    "warmup_step": [3000],
}

MIN_VRAM_MB = 2000
MAX_UTILIZATION = 100
CHECK_INTERVAL = 30

best_results = run_grid_search(base_config=CONFIG, search_space=SPACE, min_vram_mb=MIN_VRAM_MB, max_utilization=MAX_UTILIZATION)
best_results

2026-03-18 19:18:30,103 | INFO | ==========================================================================
2026-03-18 19:18:30,105 | INFO | Started Grid Search for 'sulfoxonium_seed68909_parameters'. Total experiments: 4
2026-03-18 19:18:30,106 | INFO | ==========================================================================
2026-03-18 19:18:30,107 | INFO | --- Preparing Experiment 0/3 ---
2026-03-18 19:18:30,108 | INFO | Parameters: {'train_ratio': 0.7, 'valid_ratio': 0.3, 'seed': 42, 'tag': 'seed42', 'learning_rate': 0.4, 'batch_size': 32, 'warmup_step': 3000}
2026-03-18 19:18:30,139 | INFO | Allocated GPU 0 for experiment 0.
2026-03-18 19:18:30,140 | INFO | Config file saved at: ./reproduction/7_finetune_results/sulfoxonium_seed68909_parameters/grid_configs/exp0_20260318_191830.json
2026-03-18 19:18:30,141 | INFO | To reproduce this run manually, use:
2026-03-18 19:18:30,142 | INFO | $  python train_model.py --config_json ./reproduction/7_finetune_results/sulfoxonium_seed68909_pa

,exp_id,status,test_mae,test_r2,timestamp,train_ratio,valid_ratio,seed,tag,learning_rate,batch_size,warmup_step
0,2,Success,0.070161,0.876865,20260318_205814,0.7,0.3,42,seed42,0.5,32,3000
1,0,Success,0.073438,0.862437,20260318_191830,0.7,0.3,42,seed42,0.4,32,3000
2,3,Success,0.112319,0.721435,20260318_214946,0.7,0.3,42,seed42,0.5,64,3000
3,1,Success,0.131794,0.586610,20260318_201228,0.7,0.3,42,seed42,0.4,64,3000


In [8]:
divider = "- " * 55

print("\n=== Grid Search Top Results (Sorted by Test MAE) ===\n")


for idx, row in best_results.iterrows():
    exp_id = row["exp_id"]
    status = row["status"]
    test_mae = row["test_mae"]
    test_r2 = row["test_r2"]
    timestamp = row["timestamp"]

    print(f"exp_id: {exp_id}, test_mae: {test_mae}, test_r2: {test_r2}, timestamp: {timestamp}, status: {status}")

    print(divider)


=== Grid Search Top Results (Sorted by Test MAE) ===

exp_id: 2, test_mae: 0.0701611265540123, test_r2: 0.8768650698643049, timestamp: 20260318_205814, status: Success
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
exp_id: 0, test_mae: 0.0734379962086677, test_r2: 0.8624370228585657, timestamp: 20260318_191830, status: Success
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
exp_id: 3, test_mae: 0.1123186126351356, test_r2: 0.7214352353243593, timestamp: 20260318_214946, status: Success
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
exp_id: 1, test_mae: 0.1317944228649139, test_r2: 0.5866097755293729, timestamp: 20260318_201228, status: Success
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
